# ***1- Gravação de audio com python***

In [22]:
# Importa as ferramentas necessárias
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode
# código JavaScript vai rodar
RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
    const reader = new FileReader()
    reader.onloadend = e => resolve(e.srcElement.result)
    reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
    stream = await navigator.mediaDevices.getUserMedia({ audio: true })
    recorder = new MediaRecorder(stream)
    chunks = []
    recorder.ondataavailable = e => chunks.push(e.data)
    recorder.start()
    await sleep(time)
    recorder.onstop = async ()=>{
        blob = new Blob(chunks)
        text = await b2text(blob)
        resolve(text)
    }
    recorder.stop()
})
"""

# Função que chama o JavaScript lá de cima e salva o áudio num arquivo .wav
def record(sec=5):
    display(Javascript(RECORD))
    js_result = output.eval_js('record(%s)' % (sec * 1000))
    audio = b64decode(js_result.split(',')[1])
    file_name = 'request_audio.wav'
    with open(file_name, 'wb') as f:
        f.write(audio)
    return f'/content/{file_name}'

# Começo de execução
print("Gravando... Fale algo nos próximos 5 segundos")
arquivo_audio = record(5)

print("Reproduzindo o que você gravou:")
display(Audio(arquivo_audio, autoplay=True))

Gravando... Fale algo nos próximos 5 segundos


<IPython.core.display.Javascript object>

Reproduzindo o que você gravou:


# ***2. Reconhecimento de Fala com Whisper***

In [23]:
# Instala o Whisper direto do GitHub (só precisa rodar uma vez)
!pip install git+https://github.com/openai/whisper.git -q

# Importa a biblioteca do Whisper
import whisper

# Carrega o modelo definido
modelo = whisper.load_model("small")

# Transcreve o áudio gravado na célula 1
resultado = modelo.transcribe("/content/request_audio.wav", language='pt')

# Pega só o texto da transcrição e guarda na variável 'transcription'
transcription = resultado["text"]

# Mostra na tela o que foi dito
print("\n Você disse:")
print(transcription)

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



 Você disse:
 como a IA pode ajudar a reduzir riscos financeiros.


# ***3. Integração com a API do ChatGPT***

In [26]:
# Instala a biblioteca da OpenAI
!pip install openai -q

# Importa o cliente novo
from openai import OpenAI
from getpass import getpass

# Pede a chave na hora da execução
print("Para usar o ChatGPT, você precisa da sua chave da API OpenAI")
api_key = getpass("Cole sua chave aqui: ")
cliente = OpenAI(api_key=api_key)

# Envia o texto que o Whisper transcreveu para o ChatGPT
resposta = cliente.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": transcription}]
)

# Extrai só o texto da resposta
chatgpt_response = resposta.choices[0].message.content

# Mostra a resposta na tela
print("\n ChatGPT respondeu:")
print(chatgpt_response)

Para usar o ChatGPT, você precisa da sua chave da API OpenAI
Cole sua chave aqui: ··········

 ChatGPT respondeu:
A inteligência artificial (IA) pode ajudar a reduzir riscos financeiros de diversas maneiras, tais como:

1. Análise de dados: A IA pode analisar grandes volumes de dados em tempo real e identificar padrões que indicam possíveis riscos financeiros. Isso permite às empresas tomar decisões mais informadas e proativas para mitigar esses riscos.

2. Previsão e modelagem de riscos: A IA pode ser usada para criar modelos preditivos e simulações que avaliam potenciais cenários de risco e suas probabilidades. Isso ajuda as empresas a antecipar possíveis ameaças financeiras e implementar estratégias preventivas.

3. Monitoramento de transações: A IA pode ser utilizada para detectar padrões incomuns ou atividades suspeitas em transações financeiras, o que ajuda a identificar e prevenir fraudes e outros tipos de riscos financeiros.

4. Automatização de processos: A IA pode automatizar

# ***4. Simplificando a Resposta do ChatGPT Como Voz***

In [25]:
# Instala a biblioteca gTTS (Google Text-to-Speech)
!pip install gTTS -q

# Importa o gTTS
from gtts import gTTS

# Cria um objeto gTTS com o texto da resposta
tts = gTTS(text=chatgpt_response, lang='pt', slow=False)

# Salva o áudio em um arquivo .wav na pasta /content/
arquivo_resposta = "/content/resposta_audio.wav"
tts.save(arquivo_resposta)

# Reproduz o áudio automaticamente no Colab
print(" Tocando a resposta em voz:")
display(Audio(arquivo_resposta, autoplay=True))

 Tocando a resposta em voz:
